In [11]:
pip install duckdb


Note: you may need to restart the kernel to use updated packages.


In [13]:
import duckdb
import requests
import os

# Create a directory to store the downloaded files
os.makedirs('taxi_data', exist_ok=True)

# List of URLs for the taxi data files (replace with actual URLs)
urls = [
    'https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet',
    'https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-02.parquet'
    # Add more URLs as needed
]

# Download the files
for url in urls:
    filename = os.path.join('taxi_data', os.path.basename(url))
    if not os.path.exists(filename):
        print(f"Downloading {url}...")
        response = requests.get(url)
        with open(filename, 'wb') as f:
            f.write(response.content)
        print(f"Downloaded {filename}")
    else:
        print(f"{filename} already exists, skipping download")

# Connect to a local database file
con = duckdb.connect(database='taxi_data.db')

# Create a view from the downloaded parquet files
con.execute("""
    CREATE VIEW yellow_taxi_2024 AS 
    SELECT * FROM read_parquet('taxi_data/yellow_tripdata_2024-*.parquet');
""")

# Answer Question 1
print(con.execute("SELECT COUNT(*) FROM yellow_taxi_2024").fetchall())

Downloaded taxi_data\yellow_tripdata_2024-01.parquet
Downloaded taxi_data\yellow_tripdata_2024-02.parquet
[(5972150,)]


In [15]:
import os
import requests
import duckdb

# Create a directory for the data
os.makedirs('taxi_data', exist_ok=True)

# Loop to download Jan (01) through June (06)
for i in range(1, 7):
    month = f"{i:02d}"
    url = f"https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-{month}.parquet"
    filename = os.path.join('taxi_data', f"yellow_tripdata_2024-{month}.parquet")
    
    if not os.path.exists(filename):
        print(f"Downloading {url}...")
        response = requests.get(url)
        with open(filename, 'wb') as f:
            f.write(response.content)  
# Changed from row.content to response.content

In [16]:
!pip install tqdm

In [17]:
import os
import requests
from tqdm.notebook import tqdm  # Progress bar library

# Create directory
os.makedirs('taxi_data', exist_ok=True)

# Loop to download Jan (01) through June (06)
for i in range(1, 7):
    month = f"{i:02d}"
    url = f"https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-{month}.parquet"
    filename = os.path.join('taxi_data', f"yellow_tripdata_2024-{month}.parquet")
    
    if not os.path.exists(filename):
        print(f"Starting {filename}...")
        
        # stream=True prevents loading the whole file into RAM at once
        response = requests.get(url, stream=True)
        total_size = int(response.headers.get('content-length', 0))
        
        # Write to file in chunks (1KB at a time) with a progress bar
        with open(filename, 'wb') as f, tqdm(
            desc=filename,
            total=total_size,
            unit='iB',
            unit_scale=True,
            unit_divisor=1024,
        ) as bar:
            for chunk in response.iter_content(chunk_size=1024):
                size = f.write(chunk)
                bar.update(size)
    else:
        print(f"{filename} already exists.")

taxi_data\yellow_tripdata_2024-01.parquet already exists.
taxi_data\yellow_tripdata_2024-02.parquet already exists.
taxi_data\yellow_tripdata_2024-03.parquet already exists.
taxi_data\yellow_tripdata_2024-04.parquet already exists.
taxi_data\yellow_tripdata_2024-05.parquet already exists.
taxi_data\yellow_tripdata_2024-06.parquet already exists.


In [20]:
#Question 1.
import duckdb
import os
import time

# Connect to the database (or create it if it doesn't exist)
con = duckdb.connect(database='taxi_data.db')

# First drop the existing view if it exists
con.execute("DROP VIEW IF EXISTS yellow_taxi_2024;")

# List all available parquet files and filter out the corrupted one
parquet_files = [f for f in os.listdir('taxi_data') 
                if f.startswith('yellow_tripdata_2024-') and 
                f.endswith('.parquet') and 
                not f == 'yellow_tripdata_2024-03.parquet']

# Create a comma-separated list of valid files with their paths
valid_files = [f"'taxi_data/{file}'" for file in parquet_files]
files_string = ", ".join(valid_files)

# Create the main table 'yellow_taxi_2024' from valid parquet files
con.execute(f"""
    CREATE OR REPLACE TABLE yellow_taxi_2024 AS 
    SELECT * FROM read_parquet([{files_string}]);
""")

print("Setup complete! Table 'yellow_taxi_2024' created with valid files only.")
print(f"Note: 'yellow_tripdata_2024-03.parquet' was skipped as it appears to be corrupted.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Setup complete! Table 'yellow_taxi_2024' created with valid files only.
Note: 'yellow_tripdata_2024-03.parquet' was skipped as it appears to be corrupted.


✅ yellow_tripdata_2024-01.parquet is good.
✅ yellow_tripdata_2024-02.parquet is good.
⚠️ Found corrupted file: yellow_tripdata_2024-03.parquet (0.00 MB). Deleting...
⬇️ Downloading yellow_tripdata_2024-03.parquet...


yellow_tripdata_2024-03.parquet:   0%|          | 0.00/57.3M [00:00<?, ?iB/s]

✅ yellow_tripdata_2024-04.parquet is good.
✅ yellow_tripdata_2024-05.parquet is good.
✅ yellow_tripdata_2024-06.parquet is good.


In [24]:
#Question 1.
import duckdb
import time

# Connect to the database (or create it if it doesn't exist)
con = duckdb.connect(database='taxi_data.db')

# First drop the existing table if it exists
con.execute("DROP TABLE IF EXISTS yellow_taxi_2024;")  # Changed from DROP VIEW to DROP TABLE

# Create the main table 'yellow_taxi_2024' from all 6 parquet files
con.execute("""
    CREATE OR REPLACE TABLE yellow_taxi_2024 AS 
    SELECT * FROM read_parquet('taxi_data/yellow_tripdata_2024-*.parquet');
""")

print("Setup complete! Table 'yellow_taxi_2024' created.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Setup complete! Table 'yellow_taxi_2024' created.


In [25]:
# Now that the table is created, let's count the rows!
q1_result = con.execute("SELECT COUNT(*) FROM yellow_taxi_2024").fetchall()
print(f"Question 1 Answer: {q1_result[0][0]:,}")

Question 1 Answer: 20,332,093


In [27]:
# Question 2: Count distinct PULocationIDs
q2_result = con.execute("SELECT COUNT(DISTINCT PULocationID) FROM yellow_taxi_2024").fetchall()
print(f"Question 2 Result (Distinct Locations): {q2_result[0][0]}")

Question 2 Result (Distinct Locations): 262


In [28]:
#Question 3: Testing for correct output
import time

# Test 1: Reading 1 Column
start = time.time()
con.execute("SELECT PULocationID FROM yellow_taxi_2024").fetchall()
print(f"Time for 1 column: {time.time() - start:.4f} seconds")

# Test 2: Reading 2 Columns
start = time.time()
con.execute("SELECT PULocationID, DOLocationID FROM yellow_taxi_2024").fetchall()
print(f"Time for 2 columns: {time.time() - start:.4f} seconds")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Time for 1 column: 13.2763 seconds


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Time for 2 columns: 13.3806 seconds


In [30]:
# Question 4: Count records with fare_amount = 0
q4_result = con.execute("SELECT COUNT(*) FROM yellow_taxi_2024 WHERE fare_amount = 0").fetchall()
print(f"Question 4 output: {q4_result[0][0]:,}")

Question 4 output: 8,333


In [31]:
import time

# Question 6: Retrieve distinct VendorIDs for the period 2024-03-01 to 2024-03-15
# This logic demonstrates the performance benefits of partitioning by tpep_dropoff_datetime
query_q6 = """
    SELECT DISTINCT VendorID
    FROM yellow_taxi_2024
    WHERE tpep_dropoff_datetime BETWEEN '2024-03-01' AND '2024-03-15';
"""

print("Executing query for Question 6...")

# Measure execution time to simulate performance testing
start_time = time.time()
result_q6 = con.execute(query_q6).fetchall()
end_time = time.time()

execution_duration = end_time - start_time

print(f"Result: {result_q6}")
print(f"Execution Time: {execution_duration:.4f} seconds")

Executing query for Question 6...
Result: [(2,), (1,), (6,)]
Execution Time: 0.0744 seconds
